In [1]:
# !pip install -q -U "transformers>=4.45" "trl>=0.11" "peft>=0.13" \
#     "datasets>=2.20" "accelerate>=0.34" "tokenizers>=0.20"

In [2]:
import torch, transformers, trl, peft, datasets

/home/oncreative/.local/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # 'Qwen/Qwen2.5-0.5B-Instruct'

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [6]:
tokenizer.pad_token  # padding

'<|endoftext|>'

In [7]:
tokenizer.eos_token # end of sentence 

'<|im_end|>'

In [8]:
tokenizer.bos_token

In [9]:
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

In [10]:
# float64 : 64bit 구성된 _ _ _ _ _ _ _ _ .... _ _ _   12.34, 1.234, 123.4 -1000000000.00000000001     +1000000000.00000000001
# torch.float32 : precision                                                        -100000000.0000000001      +100000000.0000000001

# torch.float16 :                                                          -1000000.0000000001      +1000000.0000000001
#     _ |  _ _ _ _ _ _ _ _ |  _ _ _ _ _ _ _ _
    
# torch.bfloat16
#     _ |  _ _ _ _ _ _ _ _   _ _ _ _|  _ _ _ _                             -100000000.00001      +100000000.000001

In [11]:
model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            dtype = torch.bfloat16,
            device_map = 'auto'
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [12]:
sum(p.numel() for p in model.parameters())

1543714304

In [13]:
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('260519_ft')
TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'


In [14]:
ds = load_dataset('json', data_files = {'train' : str(TRAIN_PATH), 'validation' : str(VAL_PATH)})

In [15]:
ds['train'][0]['messages']

[{'role': 'system', 'content': '당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다.'},
 {'role': 'user', 'content': '결재 빨리 해.'},
 {'role': 'assistant', 'content': '결재 처리 가능 시점을 알려주실 수 있을까요?'}]

In [16]:
print(tokenizer.apply_chat_template(ds['train'][0]['messages'], tokenize=False, add_generation_prompt=False))

<|im_start|>system
당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다.<|im_end|>
<|im_start|>user
결재 빨리 해.<|im_end|>
<|im_start|>assistant
결재 처리 가능 시점을 알려주실 수 있을까요?<|im_end|>



In [17]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [18]:
MODEL_ID

'Qwen/Qwen2.5-1.5B-Instruct'

In [19]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [20]:
lora_cfg = LoraConfig(
        r = 16,
        lora_alpha=32,
        lora_dropout=0.05,  # 모든 파라미터를 다 학습 -> 학습이 너무 잘된다 overfitting
        bias = 'none',
        task_type = 'CAUSAL_LM',
        target_modules = [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
        ]
)

In [21]:
model = get_peft_model(model, lora_cfg)

In [22]:
model.print_trainable_parameters() # freeze

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [31]:
from trl import SFTTrainer, SFTConfig

OUPUT_DIR = 'qwen25-1.5b_polite_email_lora'

sft_cfg = SFTConfig(
    output_dir = OUPUT_DIR,
    num_train_epochs = 8,
    per_device_train_batch_size = 2, # 20 -> 16 학습, 4 val -> batch당 에러 16 -> 에러
    per_device_eval_batch_size = 2,
    bf16=True,
    
)
    

In [32]:
trainer = SFTTrainer(
    model = model,
    args = sft_cfg,
    train_dataset = ds['train'],
    eval_dataset = ds['validation'],
    processing_class = tokenizer,
)

Truncating train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

In [33]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.184867
20,2.917024
30,2.778667
40,2.592397
50,2.483002
60,2.420804


TrainOutput(global_step=64, training_loss=2.710696205496788, metrics={'train_runtime': 20.7044, 'train_samples_per_second': 6.182, 'train_steps_per_second': 3.091, 'total_flos': 67033859966976.0, 'train_loss': 2.710696205496788})

In [ ]:
sft_cfg = SFTConfig(
    output_dir = OUPUT_DIR,
    num_train_epochs = 80,
    per_device_train_batch_size = 2, # 20 -> 16 학습, 4 val -> batch당 에러 16 -> 에러
    per_device_eval_batch_size = 2,
    learning_rate = 2e-4,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'cosine',
    bf16=True,
    max_length = 512
)
trainer = SFTTrainer(
    model = model,
    args = sft_cfg,
    train_dataset = ds['train'],
    eval_dataset = ds['validation'],
    processing_class = tokenizer,
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,0.174825
20,0.151550
30,0.134823
40,0.108144
50,0.086852
60,0.089582
70,0.082731
80,0.074166
90,0.071703
100,0.072170


In [34]:
SYSTEM_MSG

'당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다.'

In [35]:
device

'cuda'

In [36]:
def generate(user_msg, max_new_tokens):
    msgs = [
        {'role' : 'system', 'content' : SYSTEM_MSG},
        {'role' : 'user', 'content' : user_msg}
    ]

    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize= True,
        add_generation_prompt = True,
        return_tensors = 'pt',
        return_dict = True
    ).to(device)

    with torch.no_grad(): # gradient 업데이트 할필요없다
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False,
            pad_token_id = tokenizer.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[1]
    return tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()

    

In [48]:
msgs = [
        {'role' : 'system', 'content' : SYSTEM_MSG},
        {'role' : 'user', 'content' : '보고서 다시 써'}
    ]

inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize= True,
        add_generation_prompt = True,
        return_tensors = 'pt',
        return_dict = True
    ).to(device)

In [49]:
inputs

{'input_ids': tensor([[151644,   8948,    198,  64795,  82528,  33704, 125149, 131000,  23573,
          53435, 137471,  36055, 126402,  23573,  23084,  84667,  32077,  53435,
          40853,  42039,  81718, 144379, 134794, 124685, 134619,  94152,  28626,
          78952,     13, 151645,    198, 151644,    872,    198, 130626,  26698,
         131170,   3315,    235,    101, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [50]:
inputs['input_ids'].shape[1]

45

In [51]:
with torch.no_grad(): # gradient 업데이트 할필요없다
    out = model.generate(
        **inputs,
        max_new_tokens = 80,
        do_sample = False,
        pad_token_id = tokenizer.eos_token_id,
    )

In [52]:
out

tensor([[151644,   8948,    198,  64795,  82528,  33704, 125149, 131000,  23573,
          53435, 137471,  36055, 126402,  23573,  23084,  84667,  32077,  53435,
          40853,  42039,  81718, 144379, 134794, 124685, 134619,  94152,  28626,
          78952,     13, 151645,    198, 151644,    872,    198, 130626,  26698,
         131170,   3315,    235,    101, 151645,    198, 151644,  77091,    198,
         130626,    198,     34,     13,    220,     16,     24,     24,     24,
             24,     24,     24,     24,     24,     24,     24,     24,     24,
             24,     24,     24,     24,     24,     24,     24,     24,     24,
             24,     24,     24,     24,     24,     24,     24,     24,     24,
             24,     24,     24,     24,     24,     24,     24,     24,     24,
             24,     24,     24,     24,     24,     24,     24,     24,     24,
             24,     24,     24,     24,     24,     24,     24,     24,     24,
             24,     24,    

In [54]:
tokenizer.decode(out[0, 45:], skip_special_tokens=True).strip()

'보고\nC. 199999999999999999999999999999999999999999999999999999999999999999999999999'